# DATN AI-SOAR — Week 3 model training

Notebook này điều phối pipeline huấn luyện đã được kiểm thử trong repo. Logic khoa học nằm trong các module `ml.*`, không nằm rải rác trong notebook.

**Ranh giới bắt buộc:** chỉ dùng `train.jsonl`, `validation.jsonl`, `dataset_manifest.json` và file kiểm tra `colab-upload.sha256`. Notebook từ chối chạy nếu phát hiện `locked_test.jsonl`; tập test khóa chỉ được mở sau khi model đã đóng băng ở cổng Week 6.

**Đầu ra:** EDA, kiểm toán leakage, benchmark Logistic Regression / Random Forest / XGBoost, threshold theo mục tiêu precision, model package và log huấn luyện được lưu về Google Drive.

## 0. Cấu hình phiên chạy

Trước khi chạy, tạo đúng thư mục Drive `MyDrive/DATN/week3-data/processed` và chỉ đặt bốn file an toàn cho Colab vào đó:

- `train.jsonl`
- `validation.jsonl`
- `dataset_manifest.json`
- `colab-upload.sha256`

Không tải dữ liệu raw, khóa/API key, `.env` hoặc tập test khóa lên Colab.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/lethinhlord/Repo.git"
BRANCH = "feature/week3-model-training"
PROJECT_DIR = Path("/content/datn-ai-soar")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/DATN/week3-data/processed")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/DATN/week3-results")
EXPECTED_DATA_FILES = ("train.jsonl", "validation.jsonl", "dataset_manifest.json")
CHECKSUM_FILE = "colab-upload.sha256"
EXPECTED_FILES = (*EXPECTED_DATA_FILES, CHECKSUM_FILE)
RANDOM_SEED = 42

print(f"Repo: {REPO_URL}")
print(f"Branch: {BRANCH}")
print(f"Data source: {DRIVE_DATA_DIR}")
print(f"Result destination: {DRIVE_OUTPUT_DIR}")

## 1. Kiểm tra runtime và lấy đúng nhánh GitHub

Khuyến nghị dùng CPU runtime; pipeline là ML dạng bảng và cấu hình XGBoost hiện dùng `tree_method=hist`, không cần GPU.

In [ ]:
import os
import platform
import subprocess

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("CPU count:", os.cpu_count())

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "--prune", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "switch", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
else:
    # Equivalent command: git clone --branch $BRANCH --single-branch $REPO_URL $PROJECT_DIR
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

os.chdir(PROJECT_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
print("Checked out branch:", branch)
print("Git commit:", commit)
assert branch == BRANCH, f"Sai nhánh: {branch}"

## 2. Cài dependency đã khóa phiên bản

Lệnh cài đặt dùng trực tiếp `requirements.lock` và `requirements-ml.lock` của repo để hạn chế sai khác giữa Colab và Ubuntu VM.

In [ ]:
%pip install -q --upgrade pip
%pip install -q -r requirements.lock -r requirements-ml.lock -r requirements-notebook.lock

In [ ]:
import matplotlib
import numpy as np
import pandas as pd
import sklearn
import xgboost

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
assert sklearn.__version__ == "1.9.1"
assert xgboost.__version__ == "3.4.1"

## 3. Gắn Google Drive và nạp partition an toàn

Cell này chỉ sao chép ba file dữ liệu được phép và file checksum. Nếu nguồn hoặc project có `locked_test.jsonl`, phiên chạy dừng ngay.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import shutil

SOURCE_LOCKED_TEST_PATH = DRIVE_DATA_DIR / "locked_test.jsonl"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
LOCKED_TEST_PATH = PROCESSED_DIR / "locked_test.jsonl"

if SOURCE_LOCKED_TEST_PATH.exists():
    raise RuntimeError(
        "Drive data folder contains locked_test.jsonl. Remove it from the Colab staging "
        "folder; do not inspect or upload the locked test during model selection."
    )

missing = [name for name in EXPECTED_FILES if not (DRIVE_DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Thiếu {missing} trong {DRIVE_DATA_DIR}. Hãy build dataset trên Ubuntu trước."
    )

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for name in EXPECTED_FILES:
    shutil.copy2(DRIVE_DATA_DIR / name, PROCESSED_DIR / name)

checksum_rows = (PROCESSED_DIR / CHECKSUM_FILE).read_text(encoding="utf-8").splitlines()
expected_hashes = {
    line.split()[1].lstrip("*"): line.split()[0]
    for line in checksum_rows
    if line.strip()
}
assert set(expected_hashes) == set(EXPECTED_DATA_FILES), expected_hashes
for name in EXPECTED_DATA_FILES:
    actual = hashlib.sha256((PROCESSED_DIR / name).read_bytes()).hexdigest()
    assert actual == expected_hashes[name], f"SHA-256 mismatch: {name}"

if LOCKED_TEST_PATH.exists():
    raise RuntimeError("Locked test boundary violated: refusing to continue.")

print("Staged files:")
for name in EXPECTED_FILES:
    path = PROCESSED_DIR / name
    print(f"- {path.name}: {path.stat().st_size:,} bytes")

## 4. Kiểm tra manifest, split và leakage trước huấn luyện

Câu hỏi kiểm toán bắt buộc: **train có chứa thông tin trực tiếp hoặc gián tiếp về bất kỳ mẫu validation/test nào không?**

Pipeline kiểm tra trùng `run_id`, `incident_id`, bản ghi giống hệt và các feature bị cấm.

In [ ]:
import json
from collections import Counter

from ml.dataset import load_feature_schema
from ml.training import exploratory_summary, leakage_audit


def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]

manifest = json.loads((PROCESSED_DIR / "dataset_manifest.json").read_text(encoding="utf-8"))
train_rows = read_jsonl(PROCESSED_DIR / "train.jsonl")
validation_rows = read_jsonl(PROCESSED_DIR / "validation.jsonl")
schema = load_feature_schema(PROJECT_DIR / "ml" / "feature_schema.json")
audit = leakage_audit(train_rows, validation_rows)

assert manifest.get("locked_test_status") == "UNOPENED", manifest
assert manifest.get("group_key") == "scenario_id", manifest
assert audit["passed"], audit
assert not ({row["run_id"] for row in train_rows} & {row["run_id"] for row in validation_rows})
train_scenarios = {row["scenario_id"] for row in train_rows}
validation_scenarios = {row["scenario_id"] for row in validation_rows}
assert not (train_scenarios & validation_scenarios)
for partition_name, rows in (("train", train_rows), ("validation", validation_rows)):
    strata = Counter((row["family"], row["label"]) for row in rows)
    required_strata = {("account", 0), ("account", 1), ("c2", 0), ("c2", 1)}
    missing_strata = required_strata - set(strata)
    assert not missing_strata, f"{partition_name} thiếu strata: {missing_strata}"
    print(partition_name, "records=", len(rows), "strata=", dict(sorted(strata.items())))

print("Dataset hash:", manifest["dataset_hash_sha256"])
print("Feature schema:", manifest["feature_schema_version"])
print("Locked test status:", manifest["locked_test_status"])
print("Leakage audit:", json.dumps(audit, indent=2))

## 5. EDA chỉ trên TRAIN

EDA chỉ dùng TRAIN để xem cân bằng nhãn, missing, khoảng giá trị và outlier. Validation không được dùng để quyết định feature hay làm sạch dữ liệu. Với bài toán cảnh báo mất cân bằng, metric chính là PR-AUC/F1 thay vì accuracy.

In [ ]:
from IPython.display import display

eda = exploratory_summary(train_rows, schema)
summary_rows = []
for family, family_stats in eda["families"].items():
    summary_rows.append(
        {
            "family": family,
            "records": family_stats["records"],
            "independent_runs": family_stats["independent_runs"],
            "benign": family_stats["benign"],
            "malicious": family_stats["malicious"],
        }
    )
display(pd.DataFrame(summary_rows))

feature_rows = []
for family, family_stats in eda["families"].items():
    for feature, stats in family_stats["features"].items():
        feature_rows.append({"family": family, "feature": feature, **stats})
display(pd.DataFrame(feature_rows))

In [ ]:
import matplotlib.pyplot as plt

for family, feature_order in schema["x-family-features"].items():
    family_rows = [row for row in train_rows if row["family"] == family]
    figure, axes = plt.subplots(1, len(feature_order), figsize=(5 * len(feature_order), 3.5))
    axes = np.atleast_1d(axes)
    for axis, feature in zip(axes, feature_order, strict=True):
        for label, color, name in ((0, "#4C78A8", "benign"), (1, "#E45756", "malicious")):
            values = [row["features"][feature] for row in family_rows if row["label"] == label]
            axis.hist(
                values,
                bins=min(15, max(3, len(set(values)))),
                alpha=0.65,
                label=name,
                color=color,
            )
        axis.set_title(f"{family}: {feature}")
        axis.set_xlabel(feature)
        axis.set_ylabel("count")
        axis.legend()
    plt.tight_layout()
    plt.show()

## 6. Chạy quality gate của repo

Hai lệnh bắt buộc trước train:

```bash
python -m ruff check .
python -m pytest -q
```

In [ ]:
subprocess.run(["python", "-m", "ruff", "check", "."], check=True)
subprocess.run(["python", "-m", "pytest", "-q"], check=True)
print("PASS: lint and test quality gate")

## 7. Train và chọn model bằng VALIDATION

Pipeline chạy tương đương lệnh chuẩn:

```bash
python -m ml.train --config ml/training_config.yaml
```

Model được chọn theo automation eligibility, PR-AUC, F1, FPR và latency; threshold nhắm precision tối thiểu 0,90. Accuracy không phải metric lựa chọn.

In [ ]:
EVIDENCE_DIR = PROJECT_DIR / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
training_command = ["python", "-m", "ml.train", "--config", "ml/training_config.yaml"]
training_result = subprocess.run(training_command, capture_output=True, text=True)
training_log = training_result.stdout + training_result.stderr
print(training_log)
(EVIDENCE_DIR / "week3-colab-training.log").write_text(training_log, encoding="utf-8")
training_result.check_returncode()

## 8. Đọc kết quả và kiểm tra tiêu chí chấp nhận

Không tuyên bố model tốt chỉ vì train thành công. Hãy đọc confidence interval, FPR, Brier, latency và trạng thái `automation_eligible` của từng family.

In [ ]:
PACKAGE_DIR = PROJECT_DIR / "ml" / "model-package"
required_outputs = (
    "model.joblib",
    "metadata.json",
    "metrics_validation.json",
    "thresholds.json",
    "eda_train.json",
    "leakage_audit.json",
)
missing_outputs = [name for name in required_outputs if not (PACKAGE_DIR / name).is_file()]
assert not missing_outputs, f"Thiếu model outputs: {missing_outputs}"

metadata = json.loads((PACKAGE_DIR / "metadata.json").read_text(encoding="utf-8"))
metrics = json.loads((PACKAGE_DIR / "metrics_validation.json").read_text(encoding="utf-8"))
thresholds = json.loads((PACKAGE_DIR / "thresholds.json").read_text(encoding="utf-8"))
package_audit = json.loads((PACKAGE_DIR / "leakage_audit.json").read_text(encoding="utf-8"))

assert metadata["locked_test_status"] == "UNOPENED"
assert metadata["dataset_hash_sha256"] == manifest["dataset_hash_sha256"]
assert package_audit["passed"]

candidate_rows = []
for family, family_result in metrics["families"].items():
    selected = family_result["selected_model"]
    for model_name, result in family_result["candidates"].items():
        values = result["metrics"]
        candidate_rows.append(
            {
                "family": family,
                "model": model_name,
                "selected": model_name == selected,
                "threshold": result["threshold"]["threshold"],
                "automation_eligible": result["threshold"]["automation_eligible"],
                "pr_auc": values["pr_auc"],
                "precision": values["precision"],
                "recall": values["recall"],
                "f1": values["f1"],
                "fpr": values["fpr"],
                "brier": values["brier"],
                "p95_ms": values["latency"]["p95_per_record_ms"],
            }
        )

display(pd.DataFrame(candidate_rows).sort_values(["family", "selected"], ascending=[True, False]))
print(json.dumps(thresholds, indent=2))
print("Model version:", metadata["model_version"])
print("Locked test status:", metadata["locked_test_status"])

## 9. Tạo báo cáo chạy và lưu model package vào Drive

Archive chứa model đã train, metadata, metric validation, EDA, leakage audit và log. Không chứa raw data hoặc locked test.

In [ ]:
import tarfile
from datetime import UTC, datetime

experiment_summary = {
    "experiment": "week3-benchmark-v1",
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "git_commit": commit,
    "dataset_hash_sha256": metadata["dataset_hash_sha256"],
    "split": "grouped by scenario_id; train/validation only",
    "random_seed": RANDOM_SEED,
    "locked_test_status": metadata["locked_test_status"],
    "selected_models": {
        family: family_result["selected_model"]
        for family, family_result in metrics["families"].items()
    },
    "automation_eligible": {
        family: result["automation_eligible"]
        for family, result in thresholds["families"].items()
    },
    "leakage_audit_passed": package_audit["passed"],
}
summary_path = EVIDENCE_DIR / "week3-colab-experiment-summary.json"
summary_path.write_text(json.dumps(experiment_summary, indent=2) + "\n", encoding="utf-8")

archive_path = Path("/content/model-package-week3.tar.gz")
with tarfile.open(archive_path, "w:gz") as archive:
    archive.add(PACKAGE_DIR, arcname="ml/model-package")
    archive.add(
        EVIDENCE_DIR / "week3-colab-training.log",
        arcname="evidence/week3-colab-training.log",
    )
    archive.add(summary_path, arcname="evidence/week3-colab-experiment-summary.json")

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
destination = DRIVE_OUTPUT_DIR / archive_path.name
shutil.copy2(archive_path, destination)
print("Saved:", destination)
print("Size:", f"{destination.stat().st_size / 1024 / 1024:.2f} MiB")
print(json.dumps(experiment_summary, indent=2))

## 10. Kết thúc và bước tiếp theo

Chỉ coi phiên train hoàn thành khi:

1. Ruff và toàn bộ pytest đều PASS.
2. Leakage audit có `passed: true`.
3. Cả `account` và `c2` đều có kết quả cho ba candidate.
4. Metadata ghi đúng dataset hash và `locked_test_status: UNOPENED`.
5. Archive xuất hiện trong `MyDrive/DATN/week3-results`.

Sau đó tải archive về Ubuntu VM, chạy `scripts/verify_week3_model.sh`, kích hoạt API ở SHADOW mode và chụp minh chứng toàn màn hình. `automation_eligible=false` không phải lỗi kỹ thuật; nó có nghĩa model chưa đạt precision gate và hệ thống phải tiếp tục ở ASSISTED/SHADOW.